# GPAW Extraction: Generating PAW Data

This notebook demonstrates how to run a GPAW DFT calculation and extract PAW ingredients for the quantum resource estimation pipeline. The exported HDF5 file is used by **all subsequent notebooks** in this series.

**Prerequisites:** A working GPAW installation (see [GPAW docs](https://wiki.fysik.dtu.dk/gpaw/)).

> **Note:** The cells below require GPAW to be installed and will run a DFT calculation. If you don't have GPAW, you can still read through this notebook to understand the extraction process. Pre-computed data files are available in `data/`.

## Setting Up the GPAW Calculation

We start with a simple test system: metallic hydrogen in the FCC structure. This is a minimal system that demonstrates the full pipeline without excessive computational cost.

In [1]:
import warnings

# GPAW triggers harmless RuntimeWarnings (divide by zero, overflow in dot/matmul)
# on Apple Silicon due to numerical edge cases in the PAW setup and LFC phases.
# These do not affect results.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="gpaw")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="numpy")

In [2]:
# Requires GPAW to be installed
from ase.build import bulk
from gpaw import GPAW
from bloch_paw.extractor import PawExtractor

# Build crystal structure
atoms = bulk('H', 'fcc', a=3.67)

# Set up GPAW calculator
h_cut = 0.30  # real-space grid spacing (A); sets planewave cutoff via ecut ~ 1/h^2
calc = GPAW(
    mode="lcao", basis='dzp', h=h_cut,
    kpts={'size': (2, 2, 2), 'gamma': True},
    xc='PBE', nbands=5,
    symmetry={'point_group': False, 'time_reversal': False}
)
atoms.calc = calc
energy = calc.get_potential_energy(atoms)
print(f"Total energy: {energy:.6f} eV")


  ___ ___ ___ _ _ _  
 |   |   |_  | | | | 
 | | | | | . | | | | 
 |__ |  _|___|_____|  25.7.0
 |___|_|             

User:   qc@mini.local
Date:   Fri Feb 27 11:20:45 2026
Arch:   arm64
Pid:    14464
CWD:    /Users/qc/pbc/bloch-paw/examples
Python: 3.12.12
gpaw:   /Users/qc/pbc/bloch-paw/.venv/lib/python3.12/site-packages/gpaw
_gpaw:  /Users/qc/pbc/bloch-paw/.venv/lib/python3.12/site-packages/
        _gpaw.cpython-312-darwin.so
ase:    /Users/qc/pbc/bloch-paw/.venv/lib/python3.12/site-packages/ase (version 3.26.0)
numpy:  /Users/qc/pbc/bloch-paw/.venv/lib/python3.12/site-packages/numpy (version 2.3.0)
scipy:  /Users/qc/pbc/bloch-paw/.venv/lib/python3.12/site-packages/scipy (version 1.15.3)
libxc:  2.x.y
units:  Angstrom and eV
cores: 1
OpenMP: False
OMP_NUM_THREADS: 1

Input parameters:
  basis: dzp
  h: 0.3
  kpts: {gamma: True,
         size: (2, 2, 2)}
  mode: lcao
  nbands: 5
  symmetry: {point_group: False,
             time_reversal: False}
  xc: PBE

System changes: positions

... initialized

Initializing position-dependent things.

Density initialized from atomic densities
                
                
                
       H        
                
                
                
                

Atomic positions and initial magnetic moments

Positions:
   0 H      0.000000    0.000000    0.000000    ( 0.0000,  0.0000,  0.0000)

Unit cell:
           periodic     x           y           z      points  spacing
  1. axis:    yes    0.000000    1.835000    1.835000     8     0.2649
  2. axis:    yes    1.835000    0.000000    1.835000     8     0.2649
  3. axis:    yes    1.835000    1.835000    0.000000     8     0.2649

  Lengths:   2.595082   2.595082   2.595082
  Angles:   60.000000  60.000000  60.000000

Effective grid spacing dv^(1/3) = 0.2890

     iter     time        total  log10-change:
                         energy   eigst   dens
iter:   1 11:20:45    -0.249864        c


iter:   2 11:20:45    -0.250095        c -1.49


iter:   3 11:20:45    -0.252471        c -1.51


iter:   4 11:20:45    -0.252471        c -3.52


iter:   5 11:20:45    -0.252471c       c -3.50


iter:   6 11:20:45    -0.252471c       c -5.23c



Converged after 6 iterations.

Dipole moment: (-0.687517, -0.687517, -0.687517) |e|*Ang

Energy contributions relative to reference atoms: (reference = -12.490162)

Kinetic:         +0.059092
Potential:       +0.189808
External:        +0.000000
XC:              -0.401707
Entropy (-ST):   -0.056234
Local:           -0.071548
SIC:             +0.000000
--------------------------
Free energy:     -0.280588
Extrapolated:    -0.252471

Showing only first 2 kpts
 Kpt  Band  Eigenvalues  Occupancy
  0     0     -8.64288    2.00000
  0     1     15.47552    0.00000

  1     0     -3.71600    1.50101
  1     1     16.93525    0.00000


Fermi level: -3.60587

No gap
No difference between direct/indirect transitions


Total energy: -0.252471 eV


### GPAW Parameter Guide

| Parameter | Value | Purpose |
|-----------|-------|--------|
| `mode="lcao"` | LCAO basis | Linear combination of atomic orbitals; faster than plane-wave mode for small systems |
| `basis='dzp'` | Double-zeta polarised | Standard balanced basis set |
| `h=0.30` | Grid spacing (A) | Controls the real-space FFT grid density; smaller $h$ = finer grid = higher planewave cutoff |
| `kpts={'size': (2,2,2), 'gamma': True}` | $\Gamma$-centred 2x2x2 mesh | Monkhorst-Pack k-point sampling of the Brillouin zone |
| `xc='PBE'` | PBE functional | Perdew-Burke-Ernzerhof GGA exchange-correlation |
| `nbands=5` | 5 Kohn-Sham bands | Number of single-particle states to compute |
| `symmetry={...}` | All symmetries off | **Critical:** see below |

### Why Turn Off Symmetry?

The quantum algorithm requires **all k-points in the full Brillouin zone** explicitly. GPAW's default behaviour exploits point-group and time-reversal symmetry to reduce the k-mesh to the irreducible wedge. We disable both:

- `'point_group': False` --- do not fold k-points by crystal point-group symmetry
- `'time_reversal': False` --- do not identify $\mathbf{k}$ with $-\mathbf{k}$

This ensures the k-mesh is closed under the $\mathbf{k} \oplus \mathbf{Q}$ folding operation used in the LCU decomposition.

## Extracting PAW Ingredients

`PawExtractor` reads the converged GPAW calculator and computes the quantities needed for the quantum resource estimation pipeline:

- $\tilde{\rho}$ --- smooth pseudo pair-density on the real-space FFT grid
- $C^a$ --- PAW on-site Coulomb correction tensors (one per atom)
- $D^a$ --- projector density matrices (one per atom)
- $h_{pq}$ --- one-body matrix elements

The threshold parameters control sparsification of these tensors --- elements below the threshold are set to zero. Tighter thresholds reduce file size at the cost of small accuracy loss.

In [3]:
# Requires GPAW to be installed
extractor = PawExtractor(calc, nbands=5,
    thr_rho=1e-3, thr_D=1e-2, thr_C=1e-2,
    thr_h=1e-5, thr_kappa=1e-5)

# Export to HDF5 --- skip two-body kappa tensor (fast path)
extractor.export_hdf5(
    filepath="../data/lcbo_2x2x2.h5",
    write_two_body=False  # Skip kappa tensor: 0.02% accuracy cost, ~46x faster
)
print("HDF5 file written successfully.")

HDF5 file written successfully.


## About `write_two_body=False`

The most expensive part of the extraction is computing the rank-8 two-body integral $\kappa_{pqrs}$. This tensor has shape $(N_k, N_b, N_k, N_b, N_k, N_b, N_k, N_b)$ --- for the 2x2x2 mesh with 5 bands, that is $8^4 \times 5^4 = 2.56 \times 10^6$ complex entries. For larger k-meshes, it becomes completely infeasible.

**What does $\kappa$ contribute?** The two-body integral $\kappa$ provides a mean-field exchange correction to the one-body eigenvalues:

$$h'(\mathbf{k})_{ij} = h(\mathbf{k})_{ij} - \frac{1}{2} \sum_{\mathbf{k}',l} \left[\kappa_{\mathbf{k}i,\mathbf{k}'l,\mathbf{k}j,\mathbf{k}'l} - 2\,\kappa_{\mathbf{k}i,\mathbf{k}j,\mathbf{k}'l,\mathbf{k}'l}\right]$$

**What happens when $\kappa$ is absent?** When `kappa_pqrs` is `None`, `OneNormCalculator.diagonalize_one_plus_two_body()` falls back to diagonalising the raw one-body matrix $h(\mathbf{k})$ directly, skipping the mean-field exchange correction.

**How much does it matter?** For the LCBO test system, the difference in $\lambda$ is only ~0.02%. This is well below other sources of error (basis set, k-mesh convergence, DFT functional).

**Recommendation:** Skip $\kappa$ for initial calculations and exploratory work. Include it (`write_two_body=True`) for publication-quality results where sub-percent accuracy in $\lambda$ matters.

## Verifying the HDF5 File

Let's check that the file was created and inspect its contents. This section only requires `h5py` (no GPAW needed).

In [4]:
import h5py
import os

filepath = "../data/lcbo_2x2x2.h5"
if os.path.exists(filepath):
    print(f"File size: {os.path.getsize(filepath) / 1e6:.1f} MB")
    print()

    def print_h5_tree(g, indent=0):
        for key in g:
            item = g[key]
            prefix = "  " * indent
            if isinstance(item, h5py.Group):
                print(f"{prefix}{key}/")
                print_h5_tree(item, indent + 1)
            elif isinstance(item, h5py.Dataset):
                print(f"{prefix}{key}: shape={item.shape}, dtype={item.dtype}")

    with h5py.File(filepath, 'r') as f:
        print("HDF5 contents:")
        print_h5_tree(f)
else:
    print(f"{filepath} not found. Run the extraction cell above (requires GPAW).")

File size: 8.2 MB

HDF5 contents:
C_tensor/
  atom_0000: shape=(5, 5, 5, 5), dtype=float64
D_tensor/
  atom_0000: shape=(8, 5, 8, 5, 5, 5), dtype=complex128
N_atoms: shape=(), dtype=int64
Npw: shape=(), dtype=int64
kmesh/
  bz/
    cart_1_perA: shape=(8, 3), dtype=float64
    reduced: shape=(8, 3), dtype=float64
  ibz/
    cart_1_perA: shape=(8, 3), dtype=float64
    reduced: shape=(8, 3), dtype=float64
    weights: shape=(8,), dtype=float64
lattice/
  A_direct_ang: shape=(3, 3), dtype=float64
  B_2pi_cart_invA: shape=(3, 3), dtype=float64
n_electrons: shape=(), dtype=int64
one_body/
  H_kikj: shape=(8, 5, 8, 5), dtype=complex128
rho_tilde/
  data: shape=(8, 5, 8, 5, 8, 8, 8), dtype=complex128
supercell_size: shape=(3,), dtype=int64


## Supercell Size and Resource Scaling

The `export_hdf5()` function accepts an optional `supercell_size` parameter `(Lx, Ly, Lz)` that records the intended supercell dimensions for resource estimation. By default this is `(1, 1, 1)`.

The supercell size $N_a = L_x \times L_y \times L_z$ determines how resource estimates scale. With a $k$-mesh of size $N_k$ on a single unit cell, a $(2 \times 2 \times 2)$ supercell would have $N_a = 8$ atoms. The resource counts scale as:

| Quantity | Scaling | Example ($N_a = 1 \to 8$) |
|----------|---------|---------------------------|
| One-norm $\lambda$ | $\sim N_a^2$ | $\times 64$ |
| Logical qubits | $\sim N_a^{1.5}$ | $\times 23$ |
| Toffoli gates | $\sim N_a^{3.5}$ | $\times 2896$ |

For a concrete example: if your base calculation uses a 1x1x1 unit cell with a 3x3x3 k-mesh (27 k-points), scaling to a 2x2x2 supercell would multiply the atom count by 8, the number of plane waves by 8 (larger real-space cell = denser G-grid), and the bands by 8 --- leading to dramatic increases in both $\lambda$ and the gate/qubit counts.

In practice, you choose the k-mesh and supercell size to balance accuracy against the computational cost of the classical DFT and the quantum resource estimates.

## Data Files for Subsequent Notebooks

This notebook generates `data/lcbo_2x2x2.h5`, which is used by all subsequent notebooks:

| Notebook | Depends on |
|----------|------------|
| **03** Reading PAW Data | `data/lcbo_2x2x2.h5` |
| **04** One-Norm Calculation | `data/lcbo_2x2x2.h5` |
| **05** Resource Estimation | `data/lcbo_2x2x2.h5` |
| **06** Full Pipeline | `data/lcbo_2x2x2.h5` (or generates its own) |

If you don't have GPAW installed, you can use a pre-computed HDF5 file from a collaborator.